# CNN from Scratch\n**10,000 points, Batch Size = 32, Epochs = 100**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import time

# 1. Load Data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 2. Use 10,000 points
num_points = 10000
x_train = x_train[:num_points].reshape(-1, 1, 28, 28) / 255.0
y_train = y_train[:num_points]

x_test = x_test[:2000].reshape(-1, 1, 28, 28) / 255.0
y_test = y_test[:2000]

# Convert labels to one-hot encoding
y_train_one_hot = to_categorical(y_train, 10)
y_test_one_hot = to_categorical(y_test, 10)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")

## Network Layers Definition

In [ ]:
from scipy.signal import correlate2d, convolve2d

class Conv2D:
    def __init__(self, in_channels, out_channels, kernel_size):
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
        self.B = np.zeros(out_channels)
        self.kernel_size = kernel_size
        
    def forward(self, X):
        self.X = X
        batch, in_c, H, W = X.shape
        out_h, out_w = H - self.kernel_size + 1, W - self.kernel_size + 1
        out = np.zeros((batch, self.W.shape[0], out_h, out_w))
        
        for b in range(batch):
            for oc in range(self.W.shape[0]):
                for ic in range(in_c):
                    out[b, oc] += correlate2d(X[b, ic], self.W[oc, ic], mode='valid')
                out[b, oc] += self.B[oc]
        return out
        
    def backward(self, d_out, lr):
        batch, out_c, out_h, out_w = d_out.shape
        _, in_c, H, W = self.X.shape
        dW = np.zeros_like(self.W)
        dB = np.zeros_like(self.B)
        dX = np.zeros_like(self.X)
        
        for b in range(batch):
            for oc in range(out_c):
                dB[oc] += np.sum(d_out[b, oc])
                for ic in range(in_c):
                    dW[oc, ic] += correlate2d(self.X[b, ic], d_out[b, oc], mode='valid')
                    dX[b, ic] += convolve2d(d_out[b, oc], self.W[oc, ic], mode='full')
                    
        np.clip(dW, -1.0, 1.0, out=dW)
        np.clip(dB, -1.0, 1.0, out=dB)
        
        self.W -= lr * dW
        self.B -= lr * dB
        return dX

class MaxPool2D:
    def __init__(self, pool_size):
        self.pool_size = pool_size
        
    def forward(self, X):
        self.X = X
        batch, c, H, W = X.shape
        p = self.pool_size
        out_h, out_w = H // p, W // p
        out = np.zeros((batch, c, out_h, out_w))
        
        for b in range(batch):
            for ch in range(c):
                for i in range(out_h):
                    for j in range(out_w):
                        out[b, ch, i, j] = np.max(X[b, ch, i*p:(i+1)*p, j*p:(j+1)*p])
        return out
        
    def backward(self, d_out, lr):
        batch, c, out_h, out_w = d_out.shape
        p = self.pool_size
        dX = np.zeros_like(self.X)
        
        for b in range(batch):
            for ch in range(c):
                for i in range(out_h):
                    for j in range(out_w):
                        window = self.X[b, ch, i*p:(i+1)*p, j*p:(j+1)*p]
                        m = np.max(window)
                        assigned = False
                        for i_w in range(p):
                            if assigned: break
                            for j_w in range(p):
                                if window[i_w, j_w] == m:
                                    dX[b, ch, i*p+i_w, j*p+j_w] = d_out[b, ch, i, j]
                                    assigned = True
                                    break
        return dX

class Flatten:
    def forward(self, X):
        self.shape = X.shape
        return X.reshape(X.shape[0], -1)
        
    def backward(self, d_out, lr):
        return d_out.reshape(self.shape)

class Dense:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features)
        self.B = np.zeros(out_features)
        
    def forward(self, X):
        self.X = X
        return np.dot(X, self.W) + self.B
        
    def backward(self, d_out, lr):
        dW = np.dot(self.X.T, d_out)
        dB = np.sum(d_out, axis=0)
        dX = np.dot(d_out, self.W.T)
        
        np.clip(dW, -1.0, 1.0, out=dW)
        np.clip(dB, -1.0, 1.0, out=dB)
        
        self.W -= lr * dW
        self.B -= lr * dB
        return dX

class ReLU:
    def forward(self, X):
        self.X = X
        return np.maximum(0, X)
        
    def backward(self, d_out, lr):
        return d_out * (self.X > 0)

def softmax(x):
    exps = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

## Training Loop

In [1]:
# Hyperparameters
epochs = 100
batch_size = 32
learning_rate = 0.005

layers = [
    Conv2D(1, 4, 3), # 28x28 -> 26x26, 4 channels
    ReLU(),
    MaxPool2D(2),    # 26x26 -> 13x13, 4 channels
    Flatten(),       # 4 * 13 * 13 = 676
    Dense(676, 10)
]

num_batches = len(x_train) // batch_size
train_losses = []

print("Training CNN from scratch...")
start_time = time.time()
for epoch in range(epochs):
    epoch_loss = 0
    correct_preds = 0
    
    indices = np.random.permutation(len(x_train))
    x_train_shuffled = x_train[indices]
    y_train_shuffled = y_train_one_hot[indices]
    
    for b in range(num_batches):
        start = b * batch_size
        end = start + batch_size
        X = x_train_shuffled[start:end]
        Y = y_train_shuffled[start:end]
        
        # Forward pass
        out = X
        for layer in layers:
            out = layer.forward(out)
            
        probs = softmax(out)
        
        # Loss
        loss = -np.mean(np.sum(Y * np.log(probs + 1e-8), axis=1))
        epoch_loss += loss
        
        # Accuracy
        preds = np.argmax(probs, axis=1)
        correct_preds += np.sum(preds == np.argmax(Y, axis=1))
        
        # Backward pass
        d_out = (probs - Y) / batch_size
        for layer in reversed(layers):
            d_out = layer.backward(d_out, learning_rate)
            
        if (b+1) % 100 == 0:
            print(f"  Batch {b+1}/{num_batches} done.")
            
    train_losses.append(epoch_loss / num_batches)
    acc = correct_preds / len(x_train)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss / num_batches:.4f} | Accuracy: {acc:.4f} | Time: {time.time() - start_time:.2f}s")

NameError: name 'Conv2D' is not defined